# Portfolio Research: a reproducible walkthrough

Monthly walk-forward portfolio construction, after trading costs. This notebook inspects the supplied completed run. To regenerate every result, run `python run_analysis.py` from the project folder.

**Retrospective, AI-assisted study.** The universe was selected with hindsight; results are hypothetical. See `METHODOLOGY.md` for the timing, fees and limitations.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'outputs/run_manifest.json').exists(), 'Open this notebook from the project root.'
manifest = json.loads((ROOT / 'outputs/run_manifest.json').read_text())
print(json.dumps(manifest, indent=2))

{
  "config": {
    "start": "2020-01-01",
    "end": "2026-09-26",
    "test_start": "2024-01-01",
    "lookback": 504,
    "cost_bps": 10.0,
    "risk_free": 0.04,
    "stock_cap": 0.15,
    "sector_cap": 0.35
  },
  "test_first": "2024-01-02",
  "test_last": "2026-09-25",
  "test_sessions": 686,
  "rebalance_count": 33,
  "snapshot_sha256": "6809c569dfa84c55747d42b9175427709f688a70eb0aac4fcbb895e854ea48fa",
  "protocol_sha256": "1951a29139841bb606df5d164927f1cbb43ae1dab04e0731c6d385eec01b16f9",
  "versions": {
    "python": "3.12.14",
    "numpy": "2.3.5",
    "pandas": "2.2.3",
    "scipy": "1.17.0",
    "scikit_learn": "1.8.0",
    "matplotlib": "3.10.8"
  }
}


## 1. Compare all rules

CAGR is compounded annual growth. Sharpe measures return above the constant 4% assumption relative to volatility; it is not a percentage.

In [2]:
performance = pd.read_csv(ROOT / 'outputs/performance_metrics.csv', index_col=0)
print(performance.round(4).to_string())

                          total_return    cagr  volatility  sharpe  sortino  max_drawdown  calmar  annual_traded_notional
strategy                                                                                                                 
Equal weight                    0.8235  0.2469      0.1273  1.4897   2.2348       -0.1620  1.5241                  1.0099
Inverse volatility              0.7116  0.2182      0.1143  1.4418   2.1516       -0.1451  1.5047                  0.9822
Min variance (sample)           0.5348  0.1704      0.1054  1.1740   1.7379       -0.0999  1.7069                  1.5532
Min variance (shrinkage)        0.5387  0.1715      0.1051  1.1860   1.7560       -0.1016  1.6876                  1.5089
Max Sharpe (shrinkage)          0.9327  0.2739      0.1526  1.4057   2.1276       -0.1839  1.4896                  3.7312
SPY                             0.6743  0.2084      0.1558  1.0411   1.5422       -0.1876  1.1113                  0.3670


![Growth and drawdowns](outputs/performance.png)

The highest return did not produce the highest Sharpe. Minimum variance reduced drawdown but also reduced return.

## 2. Audit timing

Targets use data ending one full session before execution. The next session earns the first return from the target.

In [3]:
audit = pd.read_csv(ROOT / 'outputs/timing_audit.csv', parse_dates=['return_date','execution_date','training_start','training_end'])
assert (audit.training_end < audit.execution_date).all()
assert (audit.execution_date < audit.return_date).all()
assert (audit.n_obs == audit.lookback).all()
print(audit.loc[audit.lookback == 504].head().to_string(index=False))
print('All timing assertions passed.')

return_date execution_date training_start training_end  n_obs  lookback
 2024-01-02     2023-12-29     2021-12-28   2023-12-28    504       504
 2024-02-01     2024-01-31     2022-01-27   2024-01-30    504       504
 2024-03-01     2024-02-29     2022-02-25   2024-02-28    504       504
 2024-04-01     2024-03-28     2022-03-25   2024-03-27    504       504
 2024-05-01     2024-04-30     2022-04-27   2024-04-29    504       504
All timing assertions passed.


## 3. Examine uncertainty

Paired circular block bootstrap, 1,000 draws, 21-session blocks. Intervals are conditional on the realized history and are not adjusted for multiple comparisons.

In [4]:
ci = pd.read_csv(ROOT / 'outputs/bootstrap_sharpe_differences.csv', index_col=0)
print(ci.round(3).to_string())

                          sharpe_difference  lower_95  upper_95
strategy                                                       
Equal weight                          0.000     0.000     0.000
Inverse volatility                   -0.048    -0.293     0.203
Min variance (sample)                -0.316    -1.253     0.590
Min variance (shrinkage)             -0.304    -1.227     0.577
Max Sharpe (shrinkage)               -0.084    -0.864     0.503
SPY                                  -0.449    -0.978    -0.046


## 4. Inspect every sensitivity scenario

Do not retrospectively select the most flattering result.

In [5]:
sensitivity = pd.read_csv(ROOT / 'outputs/sensitivity.csv')
print(sensitivity.groupby('strategy')[['cagr','sharpe']].agg(['min','max']).round(3).to_string())

                           cagr        sharpe       
                            min    max    min    max
strategy                                            
Equal weight              0.245  0.248  1.478  1.498
Inverse volatility        0.214  0.219  1.402  1.450
Max Sharpe (shrinkage)    0.243  0.290  1.255  1.541
Min variance (sample)     0.161  0.183  1.094  1.313
Min variance (shrinkage)  0.163  0.183  1.110  1.315
SPY                       0.208  0.209  1.037  1.044


## 5. Follow a trade

Both buys and sells count toward traded notional. Fees reduce equity; holdings then drift until the next monthly allocation.

In [6]:
trades = pd.read_csv(ROOT / 'outputs/transactions.csv')
print(trades.loc[trades.strategy == 'Max Sharpe (shrinkage)'].head().to_string(index=False))

              strategy       date  traded_notional  cost_fraction
Max Sharpe (shrinkage) 2024-01-02         0.999001       0.000999
Max Sharpe (shrinkage) 2024-02-01         0.424377       0.000424
Max Sharpe (shrinkage) 2024-03-01         0.464250       0.000464
Max Sharpe (shrinkage) 2024-04-01         0.572234       0.000572
Max Sharpe (shrinkage) 2024-05-01         0.121382       0.000121


## 6. Reproduce and challenge

Run `python -m unittest discover -s tests -v`, then `python run_analysis.py`. Read `METHODOLOGY.md` and `outputs/RESULTS.md`.

A useful next experiment would use a point-in-time universe and a genuinely new forward evaluation window. Higher historical returns alone do not establish an investment edge.